# ASCII Stream Engine (1024x720)

Notebook para enviar **RAW 1024x720** o **ASCII aproximado a 1024x720**.

**VLC (receptores):**
```
udp://@127.0.0.1:1234
```


In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Repo root:", repo_root)


In [ ]:
import os

from ascii_stream_engine import (
    EngineConfig,
    StreamEngine,
    OpenCVCameraSource,
    AsciiRenderer,
    FfmpegUdpOutput,
    FilterPipeline,
)

FONT_PATH = "/usr/share/fonts/truetype/freefont/FreeMono.ttf"
renderer = AsciiRenderer()
if os.path.exists(FONT_PATH):
    try:
        renderer = AsciiRenderer(font_path=FONT_PATH, font_size=8)
    except TypeError:
        from PIL import ImageFont
        renderer = AsciiRenderer(font=ImageFont.truetype(FONT_PATH, 8))

config = EngineConfig(host="127.0.0.1", port=1234, fps=24, grid_w=120, grid_h=40)
filters = FilterPipeline([])

engine = StreamEngine(
    source=OpenCVCameraSource(0),
    renderer=renderer,
    sink=FfmpegUdpOutput(),
    config=config,
    filters=filters,
)


In [ ]:
# MODO RAW 1024x720
engine.stop()
engine.update_config(render_mode="raw", raw_width=1024, raw_height=720)
engine.start()


In [ ]:
# MODO ASCII aproximado a 1024x720 (depende del tamaño de fuente)
from PIL import ImageFont

font = ImageFont.truetype(FONT_PATH, 8)
bbox = font.getbbox("M")
char_w = bbox[2] - bbox[0]
char_h = bbox[3] - bbox[1]

grid_w = 1024 // char_w
grid_h = 720 // char_h

engine.stop()
engine.set_renderer(AsciiRenderer(font=font))
engine.update_config(render_mode="ascii", grid_w=grid_w, grid_h=grid_h)
engine.start()

print("grid:", grid_w, grid_h, "output:", grid_w * char_w, grid_h * char_h)


In [ ]:
# detener
engine.stop()
